# BDC Satria Data 2026 — SigLIP2-Base (Fine-tuning biasa, BUKAN frozen+LogReg)

- **Model:** `vit_base_patch16_siglip_224.v2_webli` — ~93M params, backbone SigLIP2 (kontrastif image-text + self-distillation, pretraining WebLI dengan resep v2), dipakai di sini dengan cara **fine-tuning biasa** (backbone freeze + head baru dilatih via backprop), **sama persis** dengan semua model kamu yang lain — bukan pendekatan frozen-encoder+LogReg yang dipakai dosen kamu.
- **Tujuan:** perbandingan apple-to-apple — biar bisa lihat langsung mana yang lebih tinggi, fine-tuning biasa vs cara dosen (frozen+LogReg, hasil dosen 0.9747-0.9768).
- **Resolusi:** 224 (native SigLIP2-Base ini), arsitektur ViT standar (BUKAN RoPE seperti EVA-02) — jadi nggak perlu `dynamic_img_size`.

**Catatan penting:** checkpoint SigLIP2 aslinya **tidak** punya classifier head bawaan (dia dilatih buat contrastive image-text matching, bukan klasifikasi ImageNet). Jadi saat kita panggil `timm.create_model(..., num_classes=3)`, timm otomatis **buat head baru dari nol** (random init) — persis seperti model lain yang sudah kamu coba, cuma bedanya backbone-nya berasal dari pretraining kontrastif SigLIP2, bukan supervised/MIM.

**Konsisten dgn notebook eksplorasi lain:** `StratifiedShuffleSplit` 3-fold 90/10, class weight balanced per fold, macro F1, tracking misclassified → `.xlsx` 2 sheet. Path & batch size mengikuti pola komputer kampus (model ringan, res 224).


## 1. Setup & Konfigurasi

In [1]:
from __future__ import annotations
import os
import json
import random
import time
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from PIL import Image

try:
    import timm
except ImportError:
    raise ImportError("timm belum terinstall. Jalankan: pip install timm")

In [2]:
# === PATH DATASET ===
DATA_ROOT = Path(r"D:\Downloads\BDC2026")
TRAIN_DIR = DATA_ROOT / "train"
MODELS_DIR = DATA_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = DATA_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

CLASS_FOLDERS = ["0_Recyclable", "1_Electronic", "2_Organic"]
CLASS_NAMES = ["Recyclable", "Electronic", "Organic"]
NUM_CLASSES = len(CLASS_NAMES)

# === MODEL ===
MODEL_KEY = "siglip2_base"
TIMM_NAME = "vit_base_patch16_siglip_224.v2_webli"
IMG_SIZE = 224           # native resolution SigLIP2-Base

# === CONFIG TRAINING (disesuaikan utk laptop, VRAM 4.3GB) ===
BATCH_SIZE = 48           # nilai teruji di GPU 4.3GB laptop (head-only, VRAM masih longgar di 224)
GRAD_ACCUM_STEPS = 1
NUM_EPOCHS = 15
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 4
N_FOLDS = 3
FOLD_VAL_FRAC = 0.10
SEED = 42
# Benchmark di laptop ini: num_workers>0 di Windows justru MEMPERLAMBAT (overhead spawn/reimport
# per worker > manfaat paralel) -- num_workers=0 -> 156 img/s, num_workers=8 -> 5.8 img/s.
NUM_WORKERS = 0
PREFETCH_FACTOR = None   # tidak dipakai saat NUM_WORKERS=0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Model: {MODEL_KEY} @ res {IMG_SIZE} | effective batch {BATCH_SIZE * GRAD_ACCUM_STEPS}")


Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
VRAM: 4.3 GB
Model: siglip2_base @ res 224 | effective batch 48


## 2. Index Dataset dari Folder

In [3]:
def index_dataset(train_dir: Path, class_folders: list[str]):
    records = []
    for label_idx, folder_name in enumerate(class_folders):
        folder = train_dir / folder_name
        if not folder.is_dir():
            raise FileNotFoundError(f"Folder tidak ditemukan: {folder}")
        exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        files = [p for p in folder.iterdir() if p.suffix.lower() in exts]
        for p in files:
            records.append({"path": str(p), "label": label_idx, "class_name": CLASS_NAMES[label_idx]})
    return pd.DataFrame(records)

full_df = index_dataset(TRAIN_DIR, CLASS_FOLDERS)
print(f"Total citra ter-index: {len(full_df)}")
print(full_df["class_name"].value_counts())

Total citra ter-index: 26527
class_name
Organic       12567
Recyclable     9999
Electronic     3961
Name: count, dtype: int64


## 3. Dataset & Transform (normalisasi otomatis dari timm)

In [4]:
class TrashDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row["path"]).convert("RGB")
        except Exception:
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE))
        if self.transform:
            img = self.transform(img)
        return img, int(row["label"]), row["path"]


def build_transforms(model, img_size: int, train: bool):
    from torchvision import transforms as T
    data_cfg = timm.data.resolve_model_data_config(model)
    mean, std = data_cfg["mean"], data_cfg["std"]

    if train:
        # Langsung RandomResizedCrop dari gambar asli (bukan Resize dulu baru crop) --
        # resize ganda PIL jadi CPU bottleneck yang bikin GPU idle nunggu data.
        return T.Compose([
            T.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
            T.ToTensor(),
            T.Normalize(mean=mean, std=std),
        ])
    return T.Compose([
        T.Resize((img_size, img_size)),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])

## 4. Model Builder (backbone freeze + head-only, head dibuat baru dari nol)

In [5]:
def build_model(timm_name: str, num_classes: int):
    print(f"  Loading {timm_name} ...")
    # SigLIP2 checkpoint aslinya tidak punya classifier head (contrastive pretraining) --
    # timm otomatis bikin head baru (random init) karena num_classes=3 diberikan di sini.
    model = timm.create_model(timm_name, pretrained=True, num_classes=num_classes)

    for p in model.parameters():
        p.requires_grad = False

    head_module = None
    for attr in ["head", "classifier", "fc"]:
        if hasattr(model, attr):
            candidate = getattr(model, attr)
            if isinstance(candidate, nn.Module):
                head_module = candidate
                break
    if head_module is None:
        raise AttributeError(f"Head classifier tidak ketemu untuk {timm_name}")

    for p in head_module.parameters():
        p.requires_grad = True

    return model.to(DEVICE)


## 5. Train & Evaluate

In [6]:
scaler = GradScaler("cuda", enabled=(DEVICE == "cuda"))


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    optimizer.zero_grad()

    for step, (imgs, labels, _) in enumerate(loader):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast("cuda", enabled=(DEVICE == "cuda")):
            outputs = model(imgs)
            loss = criterion(outputs, labels) / GRAD_ACCUM_STEPS
        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        running_loss += loss.item() * GRAD_ACCUM_STEPS
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return running_loss / len(loader), macro_f1


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels, all_paths, all_probs = [], [], [], []
    for imgs, labels, paths in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast("cuda", enabled=(DEVICE == "cuda")):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        running_loss += loss.item()

        probs = F.softmax(outputs.float(), dim=1)
        preds = probs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_paths.extend(paths)
        all_probs.extend(probs.cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return running_loss / len(loader), macro_f1, np.array(all_labels), np.array(all_preds), all_paths, np.array(all_probs)

## 6. Cross Validation 3-Fold + Kumpulkan Misclassified

In [ ]:
sss = StratifiedShuffleSplit(n_splits=N_FOLDS, test_size=FOLD_VAL_FRAC, random_state=SEED)

fold_results = []
detail_records = []
eval_counter = defaultdict(int)

for fold, (train_idx, val_idx) in enumerate(sss.split(full_df, full_df["label"]), start=1):
    print(f"\n{'='*64}\n---> FOLD {fold}/{N_FOLDS} <---\n{'='*64}")
    train_df = full_df.iloc[train_idx]
    val_df = full_df.iloc[val_idx]

    weights = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=train_df["label"].values)
    class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
    print(f"  Class weights: {dict(zip(CLASS_NAMES, weights.round(3)))}")

    model = build_model(TIMM_NAME, NUM_CLASSES)
    if fold == 1:
        n_total = sum(p.numel() for p in model.parameters())
        n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Total params: {n_total:,} | Trainable (head): {n_train:,}")

    train_tf = build_transforms(model, IMG_SIZE, train=True)
    val_tf = build_transforms(model, IMG_SIZE, train=False)

    train_loader = DataLoader(TrashDataset(train_df, train_tf),
                              batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True,
                              persistent_workers=(NUM_WORKERS > 0), prefetch_factor=(PREFETCH_FACTOR if NUM_WORKERS > 0 else None))
    val_loader = DataLoader(TrashDataset(val_df, val_tf),
                            batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
                            persistent_workers=(NUM_WORKERS > 0), prefetch_factor=(PREFETCH_FACTOR if NUM_WORKERS > 0 else None))

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                  lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

    best_f1 = 0.0
    epochs_no_improve = 0
    best_path = MODELS_DIR / f"{MODEL_KEY}_fold{fold}.pth"

    for epoch in range(1, NUM_EPOCHS + 1):
        epoch_start = time.time()
        tr_loss, tr_f1 = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_f1, _, _, _, _ = evaluate(model, val_loader, criterion)
        epoch_secs = time.time() - epoch_start
        scheduler.step(val_f1)

        marker = ""
        if val_f1 > best_f1:
            best_f1 = val_f1
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_path)
            marker = " [*] best"
        else:
            epochs_no_improve += 1

        print(f"  Epoch {epoch:2d}/{NUM_EPOCHS} | train_loss {tr_loss:.4f} F1 {tr_f1:.4f} | "
              f"val_loss {val_loss:.4f} F1 {val_f1:.4f}{marker} | {epoch_secs:.1f}s/epoch")

        if epochs_no_improve >= PATIENCE:
            print(f"  Early stopping.")
            break

    model.load_state_dict(torch.load(best_path, weights_only=True))
    _, final_f1, y_true, y_pred, paths, probs = evaluate(model, val_loader, criterion)
    print(f"  >> Fold {fold} best macro F1: {best_f1:.4f} | re-eval F1: {final_f1:.4f}")

    fold_results.append({"fold": fold, "best_macro_f1": best_f1})

    for i, path in enumerate(paths):
        eval_counter[path] += 1
        if y_true[i] != y_pred[i]:
            p = probs[i]
            detail_records.append({
                "fold": fold, "filepath": path,
                "true_label": CLASS_NAMES[y_true[i]],
                "predicted_label": CLASS_NAMES[y_pred[i]],
                "confidence_pct": round(float(p[y_pred[i]]) * 100, 2),
                "true_label_confidence_pct": round(float(p[y_true[i]]) * 100, 2),
                "prob_recyclable_pct": round(float(p[0]) * 100, 2),
                "prob_electronic_pct": round(float(p[1]) * 100, 2),
                "prob_organic_pct": round(float(p[2]) * 100, 2),
            })

    del model, train_loader, val_loader, optimizer, scheduler
    torch.cuda.empty_cache()


---> FOLD 1/3 <---
  Class weights: {'Recyclable': np.float64(0.884), 'Electronic': np.float64(2.232), 'Organic': np.float64(0.704)}
  Loading vit_base_patch16_siglip_224.v2_webli ...


  Total params: 92,886,531 | Trainable (head): 2,307
  Epoch  1/15 | train_loss 0.1929 F1 0.9517 | val_loss 0.0768 F1 0.9796 [*] best | 638.4s/epoch
  Epoch  2/15 | train_loss 0.0789 F1 0.9744 | val_loss 0.0581 F1 0.9814 [*] best | 312.2s/epoch
  Epoch  3/15 | train_loss 0.0647 F1 0.9782 | val_loss 0.0515 F1 0.9843 [*] best | 312.0s/epoch
  Epoch  4/15 | train_loss 0.0596 F1 0.9794 | val_loss 0.0487 F1 0.9846 [*] best | 332.1s/epoch
  Epoch  5/15 | train_loss 0.0553 F1 0.9803 | val_loss 0.0471 F1 0.9846 [*] best | 327.4s/epoch
  Epoch  6/15 | train_loss 0.0542 F1 0.9803 | val_loss 0.0470 F1 0.9840 | 332.6s/epoch
  Epoch  7/15 | train_loss 0.0510 F1 0.9823 | val_loss 0.0466 F1 0.9849 [*] best | 327.8s/epoch
  Epoch  8/15 | train_loss 0.0499 F1 0.9821 | val_loss 0.0491 F1 0.9837 | 320.7s/epoch
  Epoch  9/15 | train_loss 0.0470 F1 0.9832 | val_loss 0.0475 F1 0.9846 | 313.6s/epoch
  Epoch 10/15 | train_loss 0.0468 F1 0.9840 | val_loss 0.0471 F1 0.9852 [*] best | 314.7s/epoch
  Epoch 11/15 

## 7. Ringkasan Hasil 3-Fold

In [ ]:
f1s = [r["best_macro_f1"] for r in fold_results]
print("Ringkasan 3-Fold CV (macro F1):")
for r in fold_results:
    print(f"  Fold {r['fold']}: {r['best_macro_f1']:.4f}")
print(f"Rata-rata macro F1: {np.mean(f1s):.4f} (+/- {np.std(f1s):.4f})")

print("\nPembanding:")
print("  ViT-B/16 SWAG        : 0.9636")
print("  EVA-02 Base          : 0.9646")
print("  BEiT-Base            : 0.9695")
print("  ConvNeXt V2-Large    : 0.9720")
print("  ViT-L/16 SWAG        : 0.9770")
print("  SwinV2-Base          : 0.9776  <- terbaik sejauh ini (fine-tuning)")
print("  SigLIP-Base (dosen, frozen+LogReg) : 0.9768  <- cara BEDA, utk konteks")

pd.DataFrame(fold_results).to_csv(OUTPUT_DIR / "siglip2_base_fold_results.csv", index=False)


## 8. Laporan Misclassified (`.xlsx`, 2 sheet)

In [ ]:
detail_df = pd.DataFrame(detail_records, columns=[
    "fold", "filepath", "true_label", "predicted_label",
    "confidence_pct", "true_label_confidence_pct",
    "prob_recyclable_pct", "prob_electronic_pct", "prob_organic_pct",
])

summary_rows = []
if len(detail_df) > 0:
    for path, g in detail_df.groupby("filepath"):
        folds_wrong = sorted(g["fold"].unique().tolist())
        summary_rows.append({
            "filepath": path,
            "true_label": g["true_label"].iloc[0],
            "folds_wrong": ",".join(map(str, folds_wrong)),
            "total_wrong": len(g),
            "total_evaluated": eval_counter[path],
            "error_rate": round(len(g) / eval_counter[path], 3) if eval_counter[path] else None,
            "avg_confidence_pct": round(g["confidence_pct"].mean(), 2),
            "avg_true_label_confidence_pct": round(g["true_label_confidence_pct"].mean(), 2),
            "avg_prob_recyclable_pct": round(g["prob_recyclable_pct"].mean(), 2),
            "avg_prob_electronic_pct": round(g["prob_electronic_pct"].mean(), 2),
            "avg_prob_organic_pct": round(g["prob_organic_pct"].mean(), 2),
        })

summary_df = pd.DataFrame(summary_rows, columns=[
    "filepath", "true_label", "folds_wrong", "total_wrong", "total_evaluated", "error_rate",
    "avg_confidence_pct", "avg_true_label_confidence_pct",
    "avg_prob_recyclable_pct", "avg_prob_electronic_pct", "avg_prob_organic_pct",
])
if len(summary_df) > 0:
    summary_df = summary_df.sort_values(["error_rate", "total_wrong"], ascending=False).reset_index(drop=True)

report_path = OUTPUT_DIR / "misclassified_report_siglip2_base.xlsx"
with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    detail_df.to_excel(writer, sheet_name="Detail", index=False)
    summary_df.to_excel(writer, sheet_name="Summary", index=False)

print(f"[SAVED] {report_path}")
print(f"  Detail  : {len(detail_df)} baris")
print(f"  Summary : {len(summary_df)} gambar unik")

## Catatan
- Kalau menjalankan di laptop: ganti `DATA_ROOT`, turunkan `BATCH_SIZE` ke 8, `NUM_WORKERS` ke 0.
- Perbandingan yang adil ke hasil dosen (0.9768) itu susah 1:1, karena caranya beda total (fine-tuning vs frozen+LogReg) — tapi ini kasih kamu gambaran apakah fine-tuning biasa bisa menyamai/mengalahkan pendekatan frozen+LogReg untuk backbone SigLIP2 yang sama.
- Untuk stacking nanti: notebook ini masih SSS + hanya simpan yang salah (mode eksplorasi).

## 9. Generate Submission (Ensemble 3-Fold, Test Set)

In [ ]:
TEST_DIR = DATA_ROOT / "test"

class TestDataset(Dataset):
    def __init__(self, paths, transform=None):
        self.paths = paths
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE))
        if self.transform:
            img = self.transform(img)
        return img, path.stem


test_paths = sorted(TEST_DIR.glob("*.jpg"), key=lambda p: int(p.stem))
print(f"Total citra test: {len(test_paths)}")

infer_model = build_model(TIMM_NAME, NUM_CLASSES)
infer_tf = build_transforms(infer_model, IMG_SIZE, train=False)

test_loader = DataLoader(TestDataset(test_paths, infer_tf),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

sum_probs = None
test_ids = None

for fold in range(1, N_FOLDS + 1):
    fold_path = MODELS_DIR / f"{MODEL_KEY}_fold{fold}.pth"
    infer_model.load_state_dict(torch.load(fold_path, weights_only=True))
    infer_model.eval()

    fold_probs = []
    ids_this_fold = []
    with torch.no_grad():
        for imgs, ids in test_loader:
            imgs = imgs.to(DEVICE)
            with autocast("cuda", enabled=(DEVICE == "cuda")):
                outputs = infer_model(imgs)
            probs = F.softmax(outputs.float(), dim=1)
            fold_probs.append(probs.cpu().numpy())
            ids_this_fold.extend(ids)

    fold_probs = np.concatenate(fold_probs, axis=0)
    if sum_probs is None:
        sum_probs = fold_probs
        test_ids = ids_this_fold
    else:
        sum_probs += fold_probs
    print(f"  Fold {fold} inference selesai.")

avg_probs = sum_probs / N_FOLDS
pred_labels = avg_probs.argmax(axis=1)

submission_df = pd.DataFrame({"id": test_ids, "label": pred_labels})
submission_df["id"] = submission_df["id"].astype(int)
submission_df = submission_df.sort_values("id").reset_index(drop=True)

submission_path = DATA_ROOT / "submission.csv"
submission_df.to_csv(submission_path, index=False)
print(f"[SAVED] {submission_path} ({len(submission_df)} baris)")
print(submission_df["label"].value_counts().rename(index=dict(enumerate(CLASS_NAMES))))

del infer_model, test_loader
torch.cuda.empty_cache()